# Unilever

### Automatización aplicación pagos Farmatodo

### Imports y configuración

In [27]:
import pandas as pd
import numpy as np

from datetime import datetime

from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import numbers

from openpyxl.styles import Alignment

### Remittance del Cliente

In [28]:
remittance = pd.read_excel(
    "Remittance_FARMATODO.xlsx",
    skiprows=6,      # arranca en la fila 7
    nrows=20,
    header=[0, 1]    # filas 7 y 8 son encabezados
)

In [29]:
# Aplanar los nombres de columnas (concatena fila 7 + fila 8)
remittance.columns = [
    str(col[0]).strip() if "Unnamed" in str(col[1]) else str(col[1]).strip()
    for col in remittance.columns
]

In [30]:
remittance = remittance[["Nro Factura", "Descripción", "Total"]]

#### Transformaciones en Remittance

In [31]:
# Tranformo Remittence
## Cada cliente manda la informacion en Excel con distintos formatos (difrente indice) y a la vez para cada cliente la cantidad de filas va a variar

remittance = remittance.rename(columns={
    "Nro Factura": "Referencia / Factura",
    "Total": "Importe de factura"
})

remittance["Importe de factura"] = pd.to_numeric(remittance["Importe de factura"], errors="coerce").round(2)

In [32]:
# Definimos las condiciones
conds = [
    remittance["Referencia / Factura"].str.startswith("PMP", na=False) & (remittance["Importe de factura"] > 0),
    ~remittance["Referencia / Factura"].str.startswith("PMP", na=False) & (remittance["Importe de factura"] > 0),
    remittance["Importe de factura"] < 0
]

# Valores según condición
choices = ["Factura", "Nota Debito", "Descuentos no asociados a FC"]

# Creamos la nueva columna
remittance["Tipo de Documento"] = np.select(conds, choices, default="")

In [33]:
# Creamos columnas vacías (si no existen)
if "Descuento" not in remittance.columns:
    remittance["Descuento"] = ""

if "Motivo del descuento" not in remittance.columns:
    remittance["Motivo del descuento"] = ""

# Definimos condiciones y valores
conds = [
    remittance["Referencia / Factura"].str.startswith("NC-DC05", na=False),
    remittance["Referencia / Factura"].str.startswith("NC-DC04", na=False),
    remittance["Referencia / Factura"].str.startswith("NC-DC06", na=False),
    remittance["Referencia / Factura"].str.contains("XKZV", na=False),
    remittance["Referencia / Factura"].str.startswith("NCF-DC", na=False),
    remittance["Referencia / Factura"].str.startswith("NC-PMP", na=False),
    remittance["Referencia / Factura"].str.startswith("CNQPMP", na=False),
    remittance["Referencia / Factura"].str.startswith("NC-100", na=False),
    remittance["Referencia / Factura"].str.startswith("NC1", na=False)
]

descuentos = [
    "CONVENIOS",        # NC-DC05  -> 657
    "DSCT PROMOCIONAL", # NC-DC04  -> 987
    "DSCT PROMOCIONAL", # NC-DC06  -> 987
    "FACT PROVEEDOR",   # XKZV     -> CSB
    "CONVENIOS",        # NCF-DC   -> 657
    "DSCT AVERIAS",     # NC-PMP   -> 206
    "PGO PDTE SOPORTE", # CNQPMP   -> 551
    "DSCT AVERIAS",     # NC-100   -> 522
    "DSCT AVERIAS"      # NC1      -> 522
]

motivos = [
    "657",  # NC-DC05
    "987",  # NC-DC04
    "987",  # NC-DC06
    "CSB",  # XKZV
    "657",  # NCF-DC
    "206",  # NC-PMP
    "551",  # CNQPMP
    "522",  # NC-100
    "522"   # NC1
]

# Asignamos Descuento y Motivo del descuento
remittance["Descuento"] = np.select(conds, descuentos, default=remittance["Descuento"])
remittance["Motivo del descuento"] = np.select(conds, motivos, default=remittance["Motivo del descuento"])


In [34]:
# Ordeno Tipo de Documento de forma descendente, para que salga como en el template
remittance = remittance.sort_values(
    by="Tipo de Documento", ascending=False
).reset_index(drop=True)

### Cartera FBL5N del Cliente

In [35]:
FBL5N = pd.read_excel(
    "FBL5N_FARMATODO.xlsx",
    sheet_name="Sheet1",
    usecols=["Document Type", "Reference", "Amount in local currency", "Reason code", "Document Number", "Text"]
)
# Agregar a la importacion de cartera FBL5N que traiga info cuando: Type = "YE" y RCd = "NRO"
# Filtrar directamente Type == RV y NRO
FBL5N = FBL5N[(FBL5N["Document Type"] == "RV") | (FBL5N["Reason code"] == "NRO")]


#FBL5N = FBL5N[FBL5N["Document Type"] == "RV"]

### Transformaciones en Cartera FBL5N

In [36]:
# Renombrar columnas
FBL5N = FBL5N.rename(columns={
    "Reference": "Referencia / Factura",
    "Amount in local currency": "importe_FBL5N"
}).reset_index(drop=True)

In [37]:
FBL5N["Referencia / Factura"] = np.where(
    FBL5N["Reason code"] == "NRO",
    FBL5N["Document Number"].astype("Int64").astype(str),  # quita el .0
    FBL5N["Referencia / Factura"]
)

In [9]:
# Este codigo transformaba el dato numerico de importe FBL5N, pero en la ultima corrida no hizo falta

# Tipifico correctamente dato importe_FBL5N
# Paso 1: eliminar puntos de miles y reemplazar coma por punto decimal

#FBL5N["importe_FBL5N"] = FBL5N["importe_FBL5N"].str.replace(".", "", regex=False)  # quita separador de miles
#FBL5N["importe_FBL5N"] = FBL5N["importe_FBL5N"].str.replace(",", ".", regex=False)  # convierte decimal a punto

# Paso 2: convertir a float
#FBL5N["importe_FBL5N"] = pd.to_numeric(FBL5N["importe_FBL5N"], errors="coerce")

### Merge de Remittance y FBL5N por Referencia / Factura (hrc_template)

In [38]:
# Cruzo Remittance y FBL5N por "Referencia / Factura"
hrc_template = pd.merge(
    remittance,
    FBL5N,
    on="Referencia / Factura",
    how="left"   # mantiene todas las filas de remittance
)

#### Transformaciones en hrc_template

In [39]:
# Inicializamos la columna como NaN
hrc_template["Diferencia"] = pd.NA

# Calculamos la diferencia solo para facturas
hrc_template.loc[hrc_template["Tipo de Documento"] == "Factura", "Diferencia"] = (
     hrc_template["importe_FBL5N"] - hrc_template["Importe de factura"]
)

In [40]:
# Filtramos filas donde Diferencia no es NA y distinta de 0 (esto cambia con respecto al Template, menos dos registros)
diferencias = hrc_template[hrc_template["Diferencia"].notna() & (hrc_template["Diferencia"] != 0)].copy()

# Creamos las nuevas filas según tus reglas
registros_diferencias_entre_remittence_cartera = pd.DataFrame({
    "Tipo de Documento": "Descuentos no asociados a FC",
    "Referencia / Factura": diferencias["Referencia / Factura"],
    "Importe de factura": diferencias["Diferencia"],
    "Pago Neto": "",  # opcional
    "Descuento": diferencias["Diferencia"].apply(
        lambda x: "MENORES VALORES" if - 20000 < x < 20000 else "A definir" # Cuando haya diferencias superiores a 20K y menores a -20k debemos asignarle ODIS y Comentario
    ),
    "Motivo del descuento": diferencias["Diferencia"].apply(
        lambda x: "WOB" if x < 0 else "384"
    )
})

# Concatenamos las nuevas filas al DataFrame original
hrc_template = pd.concat([hrc_template, registros_diferencias_entre_remittence_cartera], ignore_index=True)

In [41]:
hrc_template["Comentarios"] = np.where(
    hrc_template["Descuento"] == "MENORES VALORES",
    hrc_template["Descuento"],
    hrc_template["Referencia / Factura"].fillna("") + " " + hrc_template["Descripción"].fillna("")
)


# Agrego dato Pago Neto
hrc_template["Pago Neto"] = hrc_template["Importe de factura"]

#### Agrego dato NRO

In [42]:
# Agrego dato NRO (Nota de credito)
# Agrego registros FBL5N con Reason code = NRO que no cruzaron
nota_credito = FBL5N[FBL5N["Reason code"] == "NRO"].copy()

# Forzamos el valor de "Tipo de Documento"
nota_credito["Tipo de Documento"] = "Nota de Crédito"

# Concatenamos
hrc_template = pd.concat([hrc_template, nota_credito], ignore_index=True)

In [43]:
# Datos de Tipo de Documento = Nota de Crédito (NRO)
hrc_template.loc[
    hrc_template["Tipo de Documento"] == "Nota de Crédito",
    ["Importe de factura", "Pago Neto", "Comentarios", "Motivo del descuento"]
] = hrc_template.loc[
    hrc_template["Tipo de Documento"] == "Nota de Crédito",
    ["importe_FBL5N", "importe_FBL5N", "Text", "Reason code"]
].values

#### Limpio DF de salida

In [44]:
# Limpio las columnas con las que me voy a quedar en Template ordenadas

columnas_finales = [
    "Tipo de Documento",
    "Referencia / Factura",
    "Importe de factura",
    "Descuento",
    "Motivo del descuento",
    "Pago Neto",
    "Comentarios"
]

hrc_template = hrc_template[columnas_finales]

### Creacion de inputs para cuadros en Template

##### Dato Referencia de pago

In [45]:
# --- Paso 1: Leer la celda B7 de Remittance.xlsx ---
wb_rem = load_workbook("Remittance_FARMATODO.xlsx", data_only=True)
ws_rem = wb_rem.active  
numero_orden = ws_rem["B7"].value 

##### Dato del cliente

In [46]:
# Solo leemos la primera fila y las columnas necesarias
fbl5n = pd.read_excel(
    "FBL5N_FARMATODO.xlsx",
    usecols=["Customer", "Name 1"],
    nrows=1
)

# Extraemos los valores
id_cliente = fbl5n["Customer"].iloc[0]
nombre_cliente = fbl5n["Name 1"].iloc[0]

##### Datos del pago

In [47]:
# Fecha
fecha_pago = datetime.today().date()
# Dato importe banco
importe_FBL3N = 1
# transforma dato importe de str a numerico
#importe_FBL3N = abs(float(importe_FBL3N.replace(".", "").replace(",", ".")))

##### Datos de deposito

In [48]:
# Calculamos la suma de la columna "Pago Neto"
total_pago_neto = hrc_template["Pago Neto"].sum()

In [49]:
diferencia = total_pago_neto - importe_FBL3N

### Exportacion de archivo (Template_HRC)

##### Configuracion

In [50]:
ruta_salida = "Template_HRC_Farmatodo.xlsx"

# Exportamos con pandas, indicando hoja y posición inicial
hrc_template.to_excel(
    ruta_salida,
    index=False,
    sheet_name="Template",
    startrow=17,
    startcol=2
)

In [51]:
# Abrimos el archivo para aplicar formatos y cuadros
wb = load_workbook(ruta_salida)
ws = wb["Template"]

### Generacion de cuadros

In [52]:
# Titulos Template
ws["C2"] = "Desglose de Pago"
ws["C4"] = "CAMPOS NO EDITABLES"

In [53]:
# Cuadro REFERENCIA DE PAGO
ws["G2"] = "REFERENCIA DE PAGO"
ws["H2"] = numero_orden # --- Dato dinámico ---

In [54]:
# Cuadro Informacion clinete
ws["C6"] = "Cliente" 
ws["C8"] = "Codigo de Cliente" 
ws["D6"] = nombre_cliente # --- Dato dinámico ---
ws["D8"] = id_cliente # --- Dato dinámico ---


In [55]:
# Cuadro Datos del pago
ws["C11"] = "Referencia"
ws["C12"] = numero_orden # --- Dato dinámico ---
ws["D11"] = "Fecha"
ws["D12"] = fecha_pago # --- Dato dinámico ---
ws["E11"] = "Método de Pago"
ws["E12"] = "Transferencia"
ws["F11"] = "Valor"
ws["F12"] = importe_FBL3N # --- Dato dinámico ---

In [56]:
# Cuadro de montos
ws["F6"] = "TOTAL s/ BANCOS"
ws["G6"] = importe_FBL3N # --- Dato dinámico ---
ws["F7"] = "TOTAL s/ DETALLE"
ws["G7"] = total_pago_neto # --- Dato dinámico ---
ws["F8"] = "DIFERENCIA"
ws["G8"] = -diferencia # --- Dato dinámico ---

In [57]:
# Formato del Template:

# Datos numericos en cuadros
for cell in ["G6", "G7", "G8", "F12"]:
    ws[cell].number_format = '#,##0.00'

# Formato Tabla principal
# Columnas numéricas
num_cols = ["Importe de factura", "Pago Neto"]

for col in num_cols:
    col_idx = hrc_template.columns.get_loc(col) + 3  # startcol=2 → columna C = 3 en openpyxl
    for row in range(18, 18 + len(hrc_template) + 1):  # largo de df +1
        ws.cell(row=row, column=col_idx).number_format = '#,##0.00'
        
# Columnas de texto
## Columnas de texto formateadas y centradas (excepto 'Comentarios')
str_cols = ["Tipo de Documento", "Referencia / Factura", "Descuento", "Motivo del descuento", "Comentarios"]

for col in str_cols:
    col_idx = hrc_template.columns.get_loc(col) + 3
    for row in range(18, 18 + len(hrc_template) + 1):  # largo de df +1
        cell = ws.cell(row=row, column=col_idx)
        cell.number_format = '@'  # formato texto
        if col != "Comentarios":
            cell.alignment = Alignment(horizontal="center", vertical="center")

In [58]:
# Guardar cambios en el archivo Excel
wb.save(ruta_salida)
print(f"Archivo exportado correctamente con formato: {ruta_salida}")

Archivo exportado correctamente con formato: Template_HRC_Farmatodo.xlsx
